# Cycle Summary Export
Reads the Chinese-language Cycle Summary xlsx, translates all headers to English,
and exports user-defined CSVs based on Excel column ranges.

**Only edit Cell 2 (Configuration).** Run all cells top-to-bottom.

In [1]:
import os
import re
import pandas as pd

## Cell 2 — Configuration
Edit `INPUT_FILE`, `OUTPUT_DIR`, and the `EXPORTS` list.

### How to find your column range
Open the xlsx in Excel. Row 1 shows the Cell IDs. Each test occupies **8 columns** (cycle + 7 measurements).  
Look at the column letters for the cell you want (e.g. A–H for the first cell, I–P for the second, etc.)
and enter that range as `"col_range": "A:H"`.

### Available data columns (use in `data_columns`)
| Name | Description |
|---|---|
| `Cycle` | Cycle number |
| `Charge_Capacity_Ah` | Charge capacity (Ah) |
| `Discharge_Capacity_Ah` | Discharge capacity (Ah) |
| `Charge_Energy_Wh` | Charge energy (Wh) |
| `Discharge_Energy_Wh` | Discharge energy (Wh) |
| `Energy_Efficiency_pct` | Energy efficiency (%) |
| `Cell_Max_Charge_Temp_C` | Cell max charge temperature (°C) |
| `Cell_Max_Discharge_Temp_C` | Cell max discharge temperature (°C) |

Set `data_columns` to `None` to include all 8 columns.

### Sheet names
```
25°C 0.25P
25°C 0.5P
45°C 0.5P
60°C 0.25P
60°C 0.5P 
```

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_FILE = (
    r"C:\Users\DanWindsor\OneDrive - Peak Energy\Documents"
    r"\MCM_DataConversion\20260813_CyLT_MCM_HT-B"
    r"\20260804_Attachment 4.  Cycle  Summary.xlsx"
)

OUTPUT_DIR = (
    r"C:\Users\DanWindsor\OneDrive - Peak Energy\Documents"
    r"\MCM_DataConversion\20260813_CyLT_MCM_HT-B\CSV_Exports"
)

# ── Exports ───────────────────────────────────────────────────────────────────
# Each entry defines one output CSV.
#
# Required keys:
#   filename   : output file name (saved into OUTPUT_DIR)
#   sheet      : exact sheet name from the xlsx
#   col_range  : Excel column range covering the cell block, e.g. "A:H"
#                Span multiple adjacent blocks with a wider range: "A:P" = two cells
#
# Optional keys:
#   data_columns : list of column names to keep (None = all columns in the block)

EXPORTS = [
    # ── 25°C 0.25P ────────────────────────────────────────────────────────────
    {
        "filename":     "25C_0.25P_Cell184_A-H.csv",
        "sheet":        "25\u00b0C 0.25P",
        "col_range":    "A:H",
        "data_columns": None,          # None = all 8 columns
    },
    
    {
        "filename":     "25C_0.25P_Cell205_I-P.csv",
        "sheet":        "25\u00b0C 0.25P",
        "col_range":    "I:P",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.25P_Cell222_Q-X.csv",
        "sheet":        "25\u00b0C 0.25P",
        "col_range":    "Q:X",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.25P_Cell056_AG-AN.csv",
        "sheet":        "25\u00b0C 0.25P",
        "col_range":    "AG:AN",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.25P_Cell160_AP-AV.csv",
        "sheet":        "25\u00b0C 0.25P",
        "col_range":    "AP:AV",
        "data_columns": None,
    },
    # ── 25°C 0.5P ─────────────────────────────────────────────────────────────
    {
        "filename":     "25C_0.5P_Cell190_A-H.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "A:H",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.5P_Cell204_I-P.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "I:P",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.5P_Cell206_Q-X.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "Q:X",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.5P_Cell027_Y-AF.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "Y:AF",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.5P_Cell153_AG-AN.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "AG:AN",
        "data_columns": None,
    },

    {
        "filename":     "25C_0.5P_Cell163_AO-AV.csv",
        "sheet":        "25\u00b0C 0.5P",
        "col_range":    "AO:AV",
        "data_columns": None,
    },
    # ── 45°C 0.5P ─────────────────────────────────────────────────────────────
    {
        "filename":     "45C_0.5P_Cell207_A-H.csv",
        "sheet":        "45\u00b0C 0.5P",
        "col_range":    "A:H",
        "data_columns": None,
    },

    {
        "filename":     "45C_0.5P_Cell213_I-P.csv",
        "sheet":        "45\u00b0C 0.5P",
        "col_range":    "I:P",
        "data_columns": None,
    },

    {
        "filename":     "45C_0.5P_Cell004_Q-X.csv",
        "sheet":        "45\u00b0C 0.5P",
        "col_range":    "Q:X",
        "data_columns": None,
    },

    {
        "filename":     "45C_0.5P_Cell054_Y-AF.csv",
        "sheet":        "45\u00b0C 0.5P",
        "col_range":    "Y:AF",
        "data_columns": None,
    },
    # ── 60°C 0.25P ────────────────────────────────────────────────────────────
    {
        "filename":     "60C_0.25P_CellADV2A214_A-H.csv",
        "sheet":        "60\u00b0C 0.25P",
        "col_range":    "A:H",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.25P_CellADV217_I-P.csv",
        "sheet":        "60\u00b0C 0.25P",
        "col_range":    "I:P",
        "data_columns": None,
    },


    {
        "filename":     "60C_0.25P_CellADV2A025_Q-X.csv",
        "sheet":        "60\u00b0C 0.25P",
        "col_range":    "Q:X",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.25P_CellADV2A154_Y-AF.csv",
        "sheet":        "60\u00b0C 0.25P",
        "col_range":    "Y:AF",
        "data_columns": None,
    },


    # ── 60°C 0.5P ─────────────────────────────────────────────────────────────
    {
        "filename":     "60C_0.5P_CellADV2A201_A-H.csv",
        "sheet":        "60\u00b0C 0.5P ",   # note trailing space in original sheet name
        "col_range":    "A:H",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.5P_CellADVA221_I-P.csv",
        "sheet":        "60\u00b0C 0.5P ",   # note trailing space in original sheet name
        "col_range":    "I:P",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.5P_CellADV2A019_Y-AF.csv",
        "sheet":        "60\u00b0C 0.5P ",   # note trailing space in original sheet name
        "col_range":    "Y:AF",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.5P_CellADV2A038_AG-AN.csv",
        "sheet":        "60\u00b0C 0.5P ",   # note trailing space in original sheet name
        "col_range":    "AG:AN",
        "data_columns": None,
    },

    {
        "filename":     "60C_0.5P_CellADV2A042_AO-AV.csv",
        "sheet":        "60\u00b0C 0.5P ",   # note trailing space in original sheet name
        "col_range":    "AO:AV",
        "data_columns": None,
    },
    # ── Example: span multiple blocks, export subset of columns ───────────────
    # {
    #     "filename":     "25C_0.25P_Cells184_to_222_discharge_only.csv",
    #     "sheet":        "25\u00b0C 0.25P",
    #     "col_range":    "A:X",        # blocks A-H, I-P, Q-X  (3 cells)
    #     "data_columns": ["Cycle", "Discharge_Capacity_Ah"],
    # },
]

## Cell 3 — Helper functions (do not edit)

In [3]:
CHINESE_TO_ENGLISH = {
    "充电容量/Ah":        "Charge_Capacity_Ah",
    "放电容量/Ah":        "Discharge_Capacity_Ah",
    "充电能量/Wh":        "Charge_Energy_Wh",
    "放电能量/Wh":        "Discharge_Energy_Wh",
    "能量效率/%":         "Energy_Efficiency_pct",
    "电芯充电最高温度/°C": "Cell_Max_Charge_Temp_C",
    "电芯放电最高温度/°C": "Cell_Max_Discharge_Temp_C",
}

BLOCK_WIDTH = 8  # columns per cell: cycle + 7 measurements


def col_letter_to_idx(letters: str) -> int:
    """Convert Excel column letter(s) to a 0-based index. 'A'->0, 'Z'->25, 'AA'->26."""
    letters = letters.strip().upper()
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord('A') + 1)
    return idx - 1


def idx_to_col_letter(n: int) -> str:
    """Convert 0-based column index to Excel letter(s). 0->'A', 25->'Z', 26->'AA'."""
    result = ""
    n += 1
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result


def parse_col_range(col_range: str) -> tuple:
    """
    Parse 'A:H' or 'A-H' into (start_idx, end_idx) inclusive, 0-based.
    """
    parts = re.split(r'[:\-]', col_range.strip())
    if len(parts) != 2:
        raise ValueError(f"col_range must be like 'A:H' or 'A-H', got: '{col_range}'")
    start = col_letter_to_idx(parts[0])
    end   = col_letter_to_idx(parts[1])
    return start, end


def translate_header(h) -> str:
    if pd.isna(h):
        return ""
    h = str(h).strip()
    return CHINESE_TO_ENGLISH.get(h, h)


def extract_blocks(raw: pd.DataFrame, start_col: int, end_col: int) -> pd.DataFrame:
    """
    Extract one or more 8-column cell blocks from a raw sheet DataFrame.
    Returns a tidy long-form DataFrame with Cell_ID prepended.

    start_col / end_col : 0-based inclusive column indices from parse_col_range.
    Automatically snaps to block boundaries.
    """
    # Snap to block boundaries
    block_start = (start_col // BLOCK_WIDTH) * BLOCK_WIDTH
    block_end   = ((end_col   // BLOCK_WIDTH) + 1) * BLOCK_WIDTH - 1

    frames = []
    col = block_start
    while col <= block_end and col < raw.shape[1]:
        cell_id = raw.iloc[0, col]
        if pd.isna(cell_id):
            col += BLOCK_WIDTH
            continue
        cell_id = str(cell_id).strip()

        raw_headers = list(raw.iloc[1, col : col + BLOCK_WIDTH])
        headers = [translate_header(h) for h in raw_headers]

        block_data = raw.iloc[2:, col : col + BLOCK_WIDTH].copy()
        block_data.columns = headers
        block_data = block_data.dropna(how="all").reset_index(drop=True)

        if headers[0] != "Cycle":
            block_data = block_data.rename(columns={headers[0]: "Cycle"})

        block_data.insert(0, "Cell_ID", cell_id)
        frames.append(block_data)
        col += BLOCK_WIDTH

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

## Cell 4 — Run exports

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input file : {INPUT_FILE}")
print(f"Output dir : {OUTPUT_DIR}")
print()

xl = pd.ExcelFile(INPUT_FILE)
print(f"Sheets available: {xl.sheet_names}")
print()

sheet_cache = {}   # read each sheet only once

for export in EXPORTS:
    filename   = export["filename"]
    sheet_name = export["sheet"]
    col_range  = export["col_range"]
    data_cols  = export.get("data_columns")   # None = all

    print(f"Processing: {filename}")
    print(f"  Sheet     : '{sheet_name}'")
    print(f"  Col range : {col_range}")

    if sheet_name not in xl.sheet_names:
        print(f"  ERROR: sheet not found — skipping.")
        print()
        continue

    if sheet_name not in sheet_cache:
        sheet_cache[sheet_name] = pd.read_excel(xl, sheet_name=sheet_name, header=None)
    raw = sheet_cache[sheet_name]

    start_col, end_col = parse_col_range(col_range)
    df = extract_blocks(raw, start_col, end_col)

    if df.empty:
        print(f"  WARNING: no data extracted — check col_range and sheet name.")
        print()
        continue

    if data_cols is not None:
        keep = ["Cell_ID"] + [c for c in data_cols if c != "Cell_ID"]
        missing = [c for c in keep if c not in df.columns]
        if missing:
            print(f"  WARNING: data_columns not found: {missing}")
        df = df[[c for c in keep if c in df.columns]]

    out_path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"  Cells     : {df['Cell_ID'].unique().tolist()}")
    print(f"  Rows      : {len(df):,}")
    print(f"  Columns   : {list(df.columns)}")
    print(f"  Saved     : {out_path}")
    print()

print("All exports complete.")

Input file : C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260813_CyLT_MCM_HT-B\20260804_Attachment 4.  Cycle  Summary.xlsx
Output dir : C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260813_CyLT_MCM_HT-B\CSV_Exports

Sheets available: ['25°C 0.25P', '25°C 0.5P', '45°C 0.5P', '60°C 0.25P', '60°C 0.5P ']

Processing: 25C_0.25P_Cell184_A-H.csv
  Sheet     : '25°C 0.25P'
  Col range : A:H
  Cells     : ['A-DV2A184-0.25P']
  Rows      : 664
  Columns   : ['Cell_ID', 'Cycle', 'Charge_Capacity_Ah', 'Discharge_Capacity_Ah', 'Charge_Energy_Wh', 'Discharge_Energy_Wh', 'Energy_Efficiency_pct', 'Cell_Max_Charge_Temp_C', 'Cell_Max_Discharge_Temp_C']
  Saved     : C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260813_CyLT_MCM_HT-B\CSV_Exports\25C_0.25P_Cell184_A-H.csv

Processing: 25C_0.25P_Cell205_I-P.csv
  Sheet     : '25°C 0.25P'
  Col range : I:P
  Cells     : ['A-DV2A205-0.25P']
  Rows      : 665
  Columns   : ['

## Cell 5 — Column layout reference
Run this cell to print every sheet's block-to-column-letter mapping so you can
look up the exact `col_range` for any cell without opening Excel.

In [5]:
xl_ref = pd.ExcelFile(INPUT_FILE)
for sheet in xl_ref.sheet_names:
    raw = pd.read_excel(xl_ref, sheet_name=sheet, header=None)
    num_blocks = raw.shape[1] // BLOCK_WIDTH
    print(f"Sheet: '{sheet}'")
    print(f"  {'#':<4} {'Col range':<12} Cell ID")
    for b in range(num_blocks):
        start = b * BLOCK_WIDTH
        end   = start + BLOCK_WIDTH - 1
        cell_id = raw.iloc[0, start]
        col_str = f"{idx_to_col_letter(start)}:{idx_to_col_letter(end)}"
        print(f"  {b+1:<4} {col_str:<12} {cell_id}")
    print()

Sheet: '25°C 0.25P'
  #    Col range    Cell ID
  1    A:H          A-DV2A184-0.25P
  2    I:P          A-DV2A205-0.25P
  3    Q:X          A-DV2A222-0.25P
  4    Y:AF         B-DV2A007-0.25P
  5    AG:AN        B-DV2A056-0.25
  6    AO:AV        B-DV2A160-0.25P
  7    AW:BD        C-DV2A030-0.25P
  8    BE:BL        C-DV2A086-0.25P
  9    BM:BT        C-DV2A088--0.25P
  10   BU:CB        D-DV2B005--0.25P
  11   CC:CJ        D DV2B018 0.25P
  12   CK:CR        D DV2B038 0.25P
  13   CS:CZ        E 0001045 0.25P
  14   DA:DH        E 0001059 0.25P
  15   DI:DP        E0001073 0.25P

Sheet: '25°C 0.5P'
  #    Col range    Cell ID
  1    A:H          A-DV2A190-0.5P
  2    I:P          A-DV2A204-0.5P
  3    Q:X          A-DV2A206-0.5P
  4    Y:AF         B-DV2A027-0.5P
  5    AG:AN        B-DV2A153-0.5P
  6    AO:AV        B-DV2A163-0.5P
  7    AW:BD        C-DV2A043-0.5P
  8    BE:BL        C-DV2A087-0.5P
  9    BM:BT        D DV2B051 0.5P
  10   BU:CB        D DV2B058 0.5P
  11   CC:CJ  